<a href="https://colab.research.google.com/github/jhelsas/fuks_cem/blob/main/Draft_Cross_Entropy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
import numpy as np
import numpy.random as rndm
import scipy as sp
import matplotlib.pyplot as plt
import heapq
from scipy.stats import truncnorm


# Otimizadores via Entropia Cruzada

## Versão 1 : Calcula média e variância conforme o Elite Sample

In [3]:
def optimize_CE_v1(f, x0=None, n_in=1, bounds=None, N=100, p_elite=0.1,
                   n_iter=1000, var=20.0, alpha=0.8, rng=None):
    rng = np.random.default_rng(rng)

    # Limites: sem bounds, usa -inf/+inf (suportado nativamente pela truncnorm)
    if bounds is None:
        lo = np.full(n_in, -np.inf)
        hi = np.full(n_in, np.inf)
    else:
        bounds = np.asarray(bounds, dtype=float)      # shape (n_in, 2)
        lo, hi = bounds[:, 0], bounds[:, 1]

    medias = (np.atleast_1d(np.asarray(x0, dtype=float))
              if x0 is not None else np.zeros(n_in))
    medias = np.clip(medias, lo, hi)                  # Garante média inicial viável

    # Se 'var' representa variância, o desvio padrão inicial é sqrt(var)
    desvios = np.full(n_in, np.sqrt(var), dtype=float)

    candidates = [(medias.copy(), desvios ** 2)]
    n_elite = max(1, int(round(N * p_elite)))

    for _ in range(n_iter):
        # Limites normalizados para a escala standard da truncnorm
        a = (lo - medias) / desvios
        b = (hi - medias) / desvios

        samples_x = truncnorm.rvs(a, b, loc=medias, scale=desvios,
                                  size=(N, n_in), random_state=rng)

        scores = np.array([f(x) for x in samples_x])
        elite_xs = samples_x[np.argsort(scores)[:n_elite]]

        novas_medias = elite_xs.mean(axis=0)
        novos_desvios = elite_xs.std(axis=0)

        # Atualização suavizada para evitar colapso prematuro da variância
        medias = alpha * novas_medias + (1.0 - alpha) * medias
        desvios = alpha * novos_desvios + (1.0 - alpha) * desvios
        desvios = np.maximum(desvios, 1e-4)

        candidates.append((medias.copy(), desvios ** 2))

    return medias, desvios ** 2, candidates

In [ ]:
def f(v):
    x = v[0]
    return (x - 5)**2

## Funções de teste

In [9]:

def beale(v):
    x, y = v[0], v[1]
    return (1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2 + (2.625 - x + x*y**3)**2

def three_hump_camel(v):
    x, y = v[0], v[1]
    return 2*x**2 - 1.05*x**4 + x**6/6 + x*y + y**2

def styblinski_tang(v):
    v = np.asarray(v)
    return np.sum(v**4 - 16*v**2 + 5*v) / 2

def sphere(v):
    v = np.asarray(v)
    return np.sum(v**2)

_SHEKEL_A = np.array([[4,4,4,4],[1,1,1,1],[8,8,8,8],[6,6,6,6],[3,7,3,7],
                      [2,9,2,9],[5,5,3,3],[8,1,8,1],[6,2,6,2],[7,3.6,7,3.6]])
_SHEKEL_C = np.array([0.1,0.2,0.2,0.4,0.4,0.6,0.3,0.7,0.5,0.5])

def shekel(v):
    v = np.asarray(v)
    return -np.sum(1.0 / (_SHEKEL_C + np.sum((v - _SHEKEL_A)**2, axis=1)))

def schaffer_n4(v):
    x, y = v[0], v[1]
    num = np.cos(np.sin(np.abs(x**2 - y**2)))**2 - 0.5
    return 0.5 + num / (1 + 0.001*(x**2 + y**2))**2

def schaffer_n2(v):
    x, y = v[0], v[1]
    num = np.sin(x**2 - y**2)**2 - 0.5
    return 0.5 + num / (1 + 0.001*(x**2 + y**2))**2

def rosenbrock(v):
    v = np.asarray(v)
    return np.sum(100*(v[1:] - v[:-1]**2)**2 + (1 - v[:-1])**2)

def rastrigin(v, A=10):
    v = np.asarray(v)
    return A*len(v) + np.sum(v**2 - A*np.cos(2*np.pi*v))

def mccormick(v):
    x, y = v[0], v[1]
    return np.sin(x + y) + (x - y)**2 - 1.5*x + 2.5*y + 1

def matyas(v):
    x, y = v[0], v[1]
    return 0.26*(x**2 + y**2) - 0.48*x*y

def levi_n13(v):
    x, y = v[0], v[1]
    return (np.sin(3*np.pi*x)**2
            + (x - 1)**2 * (1 + np.sin(3*np.pi*y)**2)
            + (y - 1)**2 * (1 + np.sin(2*np.pi*y)**2))

def holder_table(v):
    x, y = v[0], v[1]
    return -np.abs(np.sin(x)*np.cos(y)*np.exp(np.abs(1 - np.sqrt(x**2 + y**2)/np.pi)))

def himmelblau(v):
    x, y = v[0], v[1]
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

def griewank(v):

    v = np.asarray(v)
    i = np.arange(1, len(v) + 1)
    return 1 + np.sum(v**2)/4000 - np.prod(np.cos(v/np.sqrt(i)))


# nome: (função, dimensão, bounds por coordenada, f* min, argmin)
# Para as funções n-dimensionais, a dimensão abaixo é só um padrão: troque à vontade.
BENCHMARKS = {
    "beale":            (beale,            2, [(-4.5, 4.5)]*2,       0.0,        [3, 0.5]),
    "three_hump_camel": (three_hump_camel, 2, [(-5, 5)]*2,           0.0,        [0, 0]),
    "styblinski_tang":  (styblinski_tang,  2, [(-5, 5)]*2,           -39.16599*2,[-2.903534]*2),
    "sphere":           (sphere,           2, [(-5.12, 5.12)]*2,     0.0,        [0, 0]),
    "shekel":           (shekel,           4, [(0, 10)]*4,           -10.5364,   [4, 4, 4, 4]),
    "schaffer_n4":      (schaffer_n4,      2, [(-100, 100)]*2,       0.292579,   [0, 1.25313]),
    "schaffer_n2":      (schaffer_n2,      2, [(-100, 100)]*2,       0.0,        [0, 0]),
    "rosenbrock":       (rosenbrock,       2, [(-5, 10)]*2,          0.0,        [1, 1]),
    "rastrigin":        (rastrigin,        2, [(-5.12, 5.12)]*2,     0.0,        [0, 0]),
    "mccormick":        (mccormick,        2, [(-1.5, 4), (-3, 4)],  -1.9133,    [-0.54719, -1.54719]),
    "matyas":           (matyas,           2, [(-10, 10)]*2,         0.0,        [0, 0]),
    "levi_n13":         (levi_n13,         2, [(-10, 10)]*2,         0.0,        [1, 1]),
    "holder_table":     (holder_table,     2, [(-10, 10)]*2,         -19.2085,   [8.05502, 9.66459]),
    "himmelblau":       (himmelblau,       2, [(-5, 5)]*2,           0.0,        [3, 2]),  # há 4 mínimos
    "griewank":         (griewank,         2, [(-600, 600)]*2,       0.0,        [0, 0]),
}

## Resultados

In [8]:
optimize_CE_v1( beale , x0= [0,0], n_in=2, bounds=[[-4.5,4.5],[-4.5,4.5]])[0]

array([3.00001539, 0.50000387])

In [10]:
for name, (f, n, bounds, fstar, xstar) in BENCHMARKS.items():
    x0 = [np.mean(b) for b in bounds]          # ou sorteie um ponto
    x = optimize_CE_v1(f, x0=x0, n_in=n, bounds=[list(b) for b in bounds])[0]
    print(f"{name:18s} f(x)={f(x):.6f}  f*={fstar:.6f}  erro={abs(f(x)-fstar):.2e}  x={np.round(x,4)}")

beale              f(x)=0.000000  f*=0.000000  erro=4.17e-10  x=[3.  0.5]
three_hump_camel   f(x)=0.000000  f*=0.000000  erro=1.90e-11  x=[-0.  0.]
styblinski_tang    f(x)=-78.332331  f*=-78.331980  erro=3.51e-04  x=[-2.9035 -2.9035]
sphere             f(x)=0.000000  f*=0.000000  erro=3.31e-11  x=[-0.  0.]
shekel             f(x)=-10.536410  f*=-10.536400  erro=9.77e-06  x=[4.0008 4.0006 3.9997 3.9995]
schaffer_n4        f(x)=0.292579  f*=0.292579  erro=3.68e-07  x=[0.     1.2531]
schaffer_n2        f(x)=0.000000  f*=0.000000  erro=5.53e-14  x=[-0.  0.]
rosenbrock         f(x)=0.000345  f*=0.000000  erro=3.45e-04  x=[0.9814 0.9632]
rastrigin          f(x)=0.000000  f*=0.000000  erro=2.13e-08  x=[0. 0.]
mccormick          f(x)=-1.913223  f*=-1.913300  erro=7.70e-05  x=[-0.5472 -1.5472]
matyas             f(x)=0.000000  f*=0.000000  erro=2.28e-12  x=[-0. -0.]
levi_n13           f(x)=0.000000  f*=0.000000  erro=1.16e-09  x=[1. 1.]
holder_table       f(x)=-19.208503  f*=-19.208500  erro=2.